<a href="https://colab.research.google.com/github/Edomario082909/AHHHHR/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# --- 1. IL TUO TOKEN ---
MY_CIVITAI_TOKEN = "23a3289904c1332953a0abcfb7504cca"

# --- 2. SETUP SISTEMA ---
!apt -y update -qq && apt -y install -qq aria2
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

# --- 3. INSTALLAZIONE COMFYUI ---
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q xformers torchsde omegaconf diffusers accelerate
!pip install -q -r requirements.txt

# Installazione Nodi Custom per gestire WanVideo, GGUF e Video
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
# Commentato temporaneamente a causa di errori di clonazione
# !git clone https://github.com/kijai/ComfyUI-WanVideo.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
!pip install -q gguf # Installa la dipendenza mancante per ComfyUI-GGUF
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
# Commentato temporaneamente a causa di errori di clonazione di WanVideo
# %cd /content/ComfyUI/custom_nodes/ComfyUI-WanVideo
# !pip install -q -r requirements.txt

# --- Creazione delle cartelle dei modelli se non esistono ---
!mkdir -p /content/ComfyUI/models/checkpoints
!mkdir -p /content/ComfyUI/models/loras
!mkdir -p /content/ComfyUI/models/vae

# --- 4. DOWNLOAD MODELLI BASE (NON-CIVITAI) ---
%cd /content/ComfyUI
# Wan 2.1 T2V 1.3B (Fondamentale per i video) - VERIFICARE IL LINK SE PERSISTE L'ERRORE!
# !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/comfyanonymous/Wan2.1_1.3B_GGUF/resolve/main/wan2.1_t2v_1.3b_f16.safetensors" -d /content/ComfyUI/models/checkpoints -o wan2.1_t2v_1.3b.safetensors
# Wan VAE - VERIFICARE IL LINK SE PERSISTE L'ERRORE!
# !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B/resolve/main/models_vae_diffusion_pytorch_model.safetensors" -d /content/ComfyUI/models/vae -o wan2.1_vae.safetensors

# --- 5. IL TUO MEGA-ELENCO PERSONALIZZATO ---
# Formato: (ID_Versione, Nome_File, Cartella_Destinazione)
my_huge_list = [
    ("2747549", "ltx_lora_fov.safetensors", "loras"),
    ("2754108", "ltx_lora_2.safetensors", "loras"),
    ("2617751", "zimage_lora_1.safetensors", "loras"),
    ("2442439", "zimage_checkpoint.safetensors", "checkpoints"),
    ("2465980", "zimage_lora_2.safetensors", "loras"),
    ("2474435", "zimage_lora_3.safetensors", "loras"),
    ("2083303", "wan_lora_1.safetensors", "loras"),
    ("2200388", "wan_lora_2.safetensors", "loras"),
    ("2475163", "zimage_lora_4.safetensors", "loras"),
    ("2477457", "zimage_lora_5.safetensors", "loras"),
    ("2581135", "zimage_lora_6.safetensors", "loras"),
    ("2273468", "wan_lora_3.safetensors", "loras"),
    ("2441730", "wan_lora_4.safetensors", "loras"),
    ("2553151", "wan_lora_5.safetensors", "loras"),
    ("2460437", "zimage_lora_7.safetensors", "loras"),
    ("2073605", "wan_lora_6.safetensors", "loras"),
    ("2739037", "sdxl_real_checkpoint.safetensors", "checkpoints"),
    ("1811054", "sdxl_lora_1.safetensors", "loras"),
    ("2752410", "ltx_checkpoint.safetensors", "checkpoints"),
    ("2376136", "wan_lora_7.safetensors", "loras"),
    ("2342652", "wan_lora_8.safetensors", "loras"),
    ("2176450", "wan_lora_9.safetensors", "loras"),
    ("2430424", "wan_lora_10.safetensors", "loras"),
    ("2738377", "grok_checkpoint.safetensors", "checkpoints")
]

print("🚀 Avvio del download di massa... Mettiti comodo.")
for version_id, filename, folder in my_huge_list:
    dest_path = f"/content/ComfyUI/models/{folder}"
    url = f"https://civitai.com/api/download/models/{version_id}?token={MY_CIVITAI_TOKEN}"
    # Aria2c scaricherà i file solo se non sono già presenti
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d {dest_path} -o {filename}

# --- 6. AVVIO E TUNNEL ---
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64
import threading, subprocess, time, re, requests

def start_tunnel():
    p = subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in iter(p.stdout.readline, b''):
        link = re.search(r"https://[a-zA-Z0-0-]+\.trycloudflare\.com", line.decode())
        if link: print(f"\n\n🟢 LINK COMFYUI: {link.group(0)}\n\n")

threading.Thread(target=start_tunnel, daemon=True).start()

%cd /content/ComfyUI
!python main.py --dont-print-server --listen 0.0.0.0 --preview-method auto

### Verifica delle cartelle dei modelli
Una volta interrotta l'esecuzione di ComfyUI, puoi eseguire le seguenti celle per verificare che i modelli siano stati scaricati nelle cartelle corrette.

In [ ]:
# Elenco dei contenuti delle cartelle dei modelli
print("Contenuto di /content/ComfyUI/models/checkpoints:")
!ls -lh /content/ComfyUI/models/checkpoints

print("\nContenuto di /content/ComfyUI/models/loras:")
!ls -lh /content/ComfyUI/models/loras

print("\nContenuto di /content/ComfyUI/models/vae:")
!ls -lh /content/ComfyUI/models/vae